In [1]:
# ================================
# RESTAURANT / SUPERMARKET
# FOOD WASTE REDUCTION PROJECT
# ALL REQUIRED IMPORTS
# ================================

# Data Processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Date & Time
from datetime import datetime, timedelta

# Holiday Features
import holidays

# API Calls (Weather, Events, etc.)
import requests

# Database
import sqlite3
from sqlalchemy import create_engine

# Encoding & Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train Test Split
from sklearn.model_selection import train_test_split

# Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Feature Importance
from sklearn.inspection import permutation_importance

# Model Save / Load
import joblib

# Streamlit Dashboard
import streamlit as st

# Warnings
import warnings
warnings.filterwarnings("ignore")



In [2]:
# ==================================
# LOAD DATASET
# ==================================

df = pd.read_csv("restaurant_supermarket_food_waste_dataset_5000.csv")

# ==================================
# BASIC INFORMATION
# ==================================

print(df.head())
print(df.info())
print(df.describe())

# ==================================
# HANDLE CATEGORICAL FEATURES
# ==================================

le_product = LabelEncoder()
le_category = LabelEncoder()

df["product_name"] = le_product.fit_transform(df["product_name"])
df["category"] = le_category.fit_transform(df["category"])

# ==================================
# SELECT FEATURES
# ==================================

X = df.drop(
    columns=[
        "date",
        "qty_sold",
        "waste_qty"
    ]
)

# Target 1
y_sales = df["qty_sold"]

# Target 2
y_waste = df["waste_qty"]

# ==================================
# TRAIN TEST SPLIT
# ==================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_sales,
    test_size=0.2,
    random_state=42
)

# ==================================
# XGBOOST MODEL
# ==================================

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

# ==================================
# PREDICTIONS
# ==================================

y_pred = model.predict(X_test)

# ==================================
# EVALUATION
# ==================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

# ==================================
# FEATURE IMPORTANCE
# ==================================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Features")
print(importance.head(15))

# ==================================
# SAVE MODEL
# ==================================

joblib.dump(
    model,
    "production_recommendation_model.pkl"
)

print("\nModel Saved Successfully")

# ==================================
# PRODUCTION RECOMMENDATION FUNCTION
# ==================================

def recommend_production(
    predicted_demand,
    current_inventory,
    safety_stock=10
):
    recommendation = (
        predicted_demand
        + safety_stock
        - current_inventory
    )

    return max(0, round(recommendation))

# Example

predicted_demand = 150
current_inventory = 35

recommended_qty = recommend_production(
    predicted_demand,
    current_inventory
)

print(
    "\nRecommended Production:",
    recommended_qty
)

# ==================================
# OPTIONAL STREAMLIT DASHBOARD
# ==================================

# streamlit run app.py

         date product_name  category  shelf_life_hours  day_of_week  \
0  2024-04-12  Pizza Slice  FastFood                24            4   
1  2025-10-23       Coffee  Beverage                12            3   
2  2025-07-14     Sandwich    Bakery                48            0   
3  2025-04-29     Sandwich    Bakery                48            1   
4  2024-02-04    Ice Cream   Dessert                72            6   

   weekend_flag  month  holiday_flag  promotion_flag  temperature_c  \
0             0      4             1               0           10.2   
1             0     10             1               1           13.9   
2             0      7             0               0           11.6   
3             0      4             0               0            3.8   
4             1      2             0               0           14.9   

   rainfall_mm  footfall  price  current_stock  produced_qty  qty_sold  \
0          4.3       446    5.0             33           142       142  